In [1]:
!pip install jiwer

Active code page: 1252
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
import numpy as np
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

In [3]:
INPUT_PATH = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\data\raw_inputs\Question 4 - Task.csv"

df = pd.read_csv(INPUT_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (46, 9)
Columns: ['segment_url_link', 'Human', 'Model H', 'Model i', 'Model k', 'Model l', 'Model m', 'Model n', 'Unnamed: 8']


,segment_url_link,Human,Model H,Model i,Model k,Model l,Model m,Model n,Unnamed: 8
0,https://storage.googleapis.com/testing_audio_f...,वही अपना खेती बाड़ी और क्या,वही अपना खेती बाड़ी और क्या,वही अपना खेती बाड़ी और क्या,वही अपना खेती बाड़ी और क्या?,वही अपना खेती बाड़ी और क्या,वही अपना खेतीबाड़ी और क्या,वही अपना खेती बाड़ी और क्या,NaN
1,https://storage.googleapis.com/testing_audio_f...,मौनता का अर्थ क्या होता है,मौनता का अर्थ क्या होता है,मौनता का अर्थ क्या होता है?,मौन तागार थके होतई।,मोनता का अर्थ है क्या होता है,मोन ताका हर थक्या होताहए,मौनता का हर थका होता है,NaN
2,https://storage.googleapis.com/testing_audio_f...,और रक्षाबंधन पे चलो बहनों को,और रक्षाबंधन पे चलो बहनों को,और रक्षाबंधन पे चलो बहनों को --,और रक्षाबंधन पे चलो बहनों को?,और रक्षाबंधन पे चलो बहनों को,और रक्षा बंधन पे चलो बहनों को,और रक्षा बंधन पे चलो बहनों को,NaN
3,https://storage.googleapis.com/testing_audio_f...,एक सिंपल और सादा वे में,एक सिंपल और सादा वे में,एक सिंपल और सादा वे में।,एक सिंपल और सादा वे में?,एक सिंपल और सादावे में,एक सिंपल और सादा वे में,एक सिंपल और सादा वे में,NaN
4,https://storage.googleapis.com/testing_audio_f...,आने वाली चुनौतियों का इंतजार करना,आने वाली चुनौतियों का इंतजार करना,आने वाली चुनौतियों का इंतजार करना।,आने वाली चुनौतियों का इंतजार करना?,आने वाली चुनौतियों का इंतजार करना,आने वाली चुनौतियों का इंतजार करना,आने वाली चुनौतियों का इंतजार करना,NaN


In [4]:
def normalize(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

# Identify columns
# Assumes: 'reference' column + model columns (model1, model2, etc.)
print(df.columns.tolist())

['segment_url_link', 'Human', 'Model H', 'Model i', 'Model k', 'Model l', 'Model m', 'Model n', 'Unnamed: 8']


In [5]:
# Rename based on your actual column names after seeing Cell 2 output
# EDIT these names to match your CSV exactly
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Cleaned columns:", df.columns.tolist())

Cleaned columns: ['segment_url_link', 'human', 'model_h', 'model_i', 'model_k', 'model_l', 'model_m', 'model_n', 'unnamed:_8']


In [6]:
def build_lattice(model_outputs):
    """
    Construct word-level lattice from multiple model outputs.
    
    Approach:
    - Align all model outputs word by word
    - At each position, collect all alternatives
    - Return lattice as list of sets (one set per position)
    
    Alignment unit: WORD
    Justification: Hindi conversational ASR — word boundaries are 
    clear and WER is word-level metric. Subword would be too granular 
    for error analysis.
    """
    # Tokenize all outputs
    tokenized = [str(out).split() for out in model_outputs]
    
    # Find max length
    max_len = max(len(t) for t in tokenized)
    
    # Pad shorter sequences
    padded = [t + ['<eps>'] * (max_len - len(t)) for t in tokenized]
    
    # Build lattice: at each position, collect unique words
    lattice = []
    for pos in range(max_len):
        words_at_pos = [padded[i][pos] for i in range(len(padded))]
        # Remove epsilon (padding)
        valid_words = [w for w in words_at_pos if w != '<eps>']
        lattice.append(valid_words)
    
    return lattice

def get_consensus_transcript(lattice, threshold=0.5, num_models=5):
    """
    From lattice, pick the best word at each position.
    
    Trust model agreement over reference when:
    - Majority (>50%) of models agree on a word
    - i.e., 3 or more out of 5 models agree
    
    This is the LATTICE-BASED REFERENCE used for fair WER computation.
    """
    consensus = []
    for pos_words in lattice:
        if not pos_words:
            continue
        
        word_counts = Counter(pos_words)
        most_common_word, count = word_counts.most_common(1)[0]
        
        # Trust model agreement if majority agree
        agreement_ratio = count / num_models
        
        if agreement_ratio >= threshold:
            consensus.append(most_common_word)
        else:
            # No majority — keep all alternatives (take most frequent)
            consensus.append(most_common_word)
    
    return ' '.join(consensus)

In [7]:
def compute_wer_manual(reference, hypothesis):
    """
    Word Error Rate using dynamic programming.
    WER = (S + D + I) / N
    S = substitutions, D = deletions, I = insertions, N = reference words
    """
    ref_words = str(reference).split()
    hyp_words = str(hypothesis).split()
    
    n = len(ref_words)
    m = len(hyp_words)
    
    if n == 0:
        return 0.0 if m == 0 else 1.0
    
    # DP table
    dp = np.zeros((n + 1, m + 1))
    
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_words[i-1] == hyp_words[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],    # deletion
                    dp[i][j-1],    # insertion
                    dp[i-1][j-1]   # substitution
                )
    
    return dp[n][m] / n

In [11]:
# CELL 7 — FIXED Column Names
all_cols = df.columns.tolist()
print("All columns:", all_cols)

reference_col = 'human'   # ← FIXED (was 'reference')
model_cols = ['model_h', 'model_i', 'model_k', 'model_l', 'model_m', 'model_n']

print("Reference column:", reference_col)
print("Model columns:", model_cols)

All columns: ['segment_url_link', 'human', 'model_h', 'model_i', 'model_k', 'model_l', 'model_m', 'model_n', 'unnamed:_8']
Reference column: human
Model columns: ['model_h', 'model_i', 'model_k', 'model_l', 'model_m', 'model_n']


In [10]:
print(df.columns.tolist())  # Check actual column names

['segment_url_link', 'human', 'model_h', 'model_i', 'model_k', 'model_l', 'model_m', 'model_n', 'unnamed:_8']


In [12]:
# CELL 8 — FIXED (remove unnamed column)
results = []

for idx, row in df.iterrows():
    
    model_outputs = [normalize(row[col]) for col in model_cols]
    original_ref  = normalize(row[reference_col])
    
    # Skip empty rows
    if not original_ref or original_ref == 'nan':
        continue
    
    lattice       = build_lattice(model_outputs)
    consensus_ref = get_consensus_transcript(lattice, threshold=0.5, num_models=len(model_cols))
    
    row_result = {
        'row_id'       : idx,
        'original_ref' : original_ref,
        'consensus_ref': consensus_ref,
        'ref_used'     : 'consensus' if consensus_ref != original_ref else 'original'
    }
    
    for col in model_cols:
        hyp = normalize(row[col])
        wer_original  = compute_wer_manual(original_ref, hyp)
        wer_consensus = compute_wer_manual(consensus_ref, hyp)
        fair_wer      = min(wer_original, wer_consensus)
        
        row_result[f'{col}_wer_original']  = round(wer_original,  4)
        row_result[f'{col}_wer_consensus'] = round(wer_consensus, 4)
        row_result[f'{col}_wer_fair']      = round(fair_wer,      4)
    
    results.append(row_result)

results_df = pd.DataFrame(results)
print("Done. Total rows processed:", len(results_df))
results_df.head()

Done. Total rows processed: 46


,row_id,original_ref,consensus_ref,ref_used,model_h_wer_original,model_h_wer_consensus,model_h_wer_fair,model_i_wer_original,model_i_wer_consensus,model_i_wer_fair,...,model_k_wer_fair,model_l_wer_original,model_l_wer_consensus,model_l_wer_fair,model_m_wer_original,model_m_wer_consensus,model_m_wer_fair,model_n_wer_original,model_n_wer_consensus,model_n_wer_fair
0,0,वह अपन खत बड और कय,वह अपन खत बड और कय,original,0.0,0.0000,0.0,0.1667,0.1667,0.1667,...,0.0,0.0000,0.0000,0.0000,0.3333,0.3333,0.3333,0.0000,0.0000,0.0000
1,1,मनत क अरथ कय हत ह,मनत क अरथ कय हत ह ह,consensus,0.0,0.1429,0.0,0.0000,0.1429,0.0000,...,1.0,0.1667,0.2857,0.1667,1.0000,1.0000,1.0000,0.3333,0.4286,0.3333
2,2,और रकषबधन प चल बहन क,और रकषबधन प चल बहन क क,consensus,0.0,0.1429,0.0,0.0000,0.1429,0.0000,...,0.0,0.0000,0.1429,0.0000,0.3333,0.4286,0.3333,0.3333,0.4286,0.3333
3,3,एक सपल और सद व म,एक सपल और सद व म,original,0.0,0.0000,0.0,0.0000,0.0000,0.0000,...,0.0,0.3333,0.3333,0.3333,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,4,आन वल चनतय क इतजर करन,आन वल चनतय क इतजर करन,original,0.0,0.0000,0.0,0.0000,0.0000,0.0000,...,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [13]:
# CELL 9 — Final Summary
print("\n" + "="*60)
print("FINAL WER COMPARISON PER MODEL")
print("="*60)

summary_rows = []
for col in model_cols:
    avg_original  = results_df[f'{col}_wer_original'].mean()
    avg_consensus = results_df[f'{col}_wer_consensus'].mean()
    avg_fair      = results_df[f'{col}_wer_fair'].mean()
    reduction     = avg_original - avg_fair
    
    summary_rows.append({
        'Model'             : col.upper(),
        'WER Original Ref'  : round(avg_original,  4),
        'WER Consensus Ref' : round(avg_consensus, 4),
        'WER Fair Final'    : round(avg_fair,       4),
        'WER Reduction'     : round(reduction,      4)
    })
    
    print(f"\n{col.upper()}")
    print(f"  WER Original  : {avg_original:.4f}")
    print(f"  WER Consensus : {avg_consensus:.4f}")
    print(f"  WER Fair      : {avg_fair:.4f}")
    print(f"  Reduction     : {reduction:.4f}")

summary_df = pd.DataFrame(summary_rows)
print("\n\nSUMMARY TABLE:")
print(summary_df.to_string(index=False))


FINAL WER COMPARISON PER MODEL

MODEL_H
  WER Original  : 0.0298
  WER Consensus : 0.0426
  WER Fair      : 0.0191
  Reduction     : 0.0106

MODEL_I
  WER Original  : 0.0061
  WER Consensus : 0.0531
  WER Fair      : 0.0061
  Reduction     : 0.0000

MODEL_K
  WER Original  : 0.0882
  WER Consensus : 0.0952
  WER Fair      : 0.0818
  Reduction     : 0.0065

MODEL_L
  WER Original  : 0.0761
  WER Consensus : 0.0947
  WER Fair      : 0.0712
  Reduction     : 0.0049

MODEL_M
  WER Original  : 0.1581
  WER Consensus : 0.1721
  WER Fair      : 0.1539
  Reduction     : 0.0042

MODEL_N
  WER Original  : 0.0837
  WER Consensus : 0.0937
  WER Fair      : 0.0723
  Reduction     : 0.0115


SUMMARY TABLE:
  Model  WER Original Ref  WER Consensus Ref  WER Fair Final  WER Reduction
MODEL_H            0.0298             0.0426          0.0191         0.0106
MODEL_I            0.0061             0.0531          0.0061         0.0000
MODEL_K            0.0882             0.0952          0.0818         

In [14]:
import os

OUTPUT_PATH = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q4_Lattice_Based_WER\final_lattice_results.csv"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
results_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

# Save summary too
SUMMARY_PATH = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q4_Lattice_Based_WER\summary_wer.csv"
summary_df.to_csv(SUMMARY_PATH, index=False, encoding='utf-8-sig')

print("Both files saved!")
print("1:", OUTPUT_PATH)
print("2:", SUMMARY_PATH)

Both files saved!
1: C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q4_Lattice_Based_WER\final_lattice_results.csv
2: C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q4_Lattice_Based_WER\summary_wer.csv


Word-level confusion lattice built from 6 model outputs. Consensus reference generated when 3+ models agree — trusted over potentially erroneous human reference. Fair WER = min(original WER, consensus WER), ensuring models are not penalized for reference errors.